In [1]:
%load_ext autoreload
%autoreload 2

import json
import random
import spglib
import numpy as np
import sympy as sp
from ase.spacegroup import Spacegroup
from ase.data import chemical_symbols
from doped.utils.symmetry import get_wyckoff_dict_from_sgn
from monty.serialization import dumpfn

### Generate random representations of crystal structures

In [2]:
def get_SpacegroupName(spacegroupNumber):
    name = Spacegroup(spacegroupNumber).symbol
    name = name.replace('-1', '1bar').replace('-6', '6bar').replace('-3', '3bar').replace('-4', '4bar')
    parts = name.split()
    parts += ['1'] * (4 - len(parts))
    spacegroupName = ' '.join(parts) + " "
    return spacegroupName

In [3]:
def get_elements():
    # elements = [symbol for symbol in chemical_symbols if symbol]
    elements = ['H', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Na', 'Mg', 'Al', 
                'Si', 'P', 'S', 'Cl', 'K', 'Ca', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 
                'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr', 
                'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 
                'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 
                'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 
                'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 
                'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At', 'Rn']
    random_element = random.choice(elements)
    return random_element

In [4]:
def get_wyckoff(spacegroupNumber):
    wyckoff_data = get_wyckoff_dict_from_sgn(spacegroupNumber)
    wyckoff = random.choice(list(wyckoff_data.keys()))
    return wyckoff, wyckoff_data

In [5]:
def get_xyz(wyckoff_data, wyckoff):
    xyz = ""
    for i in sp.symbols('x y z'):
        isbool = any(i in j.free_symbols for j in wyckoff_data[wyckoff][0])
        if isbool:
            xyz += "0."+"".join(map(str, random.choices(range(10), k=3))) + " "
        else:
            xyz += "0.000 "
    return xyz

In [6]:
def get_positions(wyckoff_data, wyckoff, xyz):
    substitutions = {sp.symbols('x'): xyz.split(" ")[0], sp.symbols('y'): xyz.split(" ")[1], sp.symbols('z'): xyz.split(" ")[2]}
    wyckoff_positions = wyckoff_data[wyckoff]
    wyckoff_positions_sub = [[float(coord.subs(substitutions)) for coord in pos] for pos in wyckoff_positions]
    wyckoff_positions_sub = np.array(wyckoff_positions_sub) % 1
    return wyckoff_positions_sub

In [7]:
def main():
    String = ""
    spacegroupNumber = random.choice(range(1, 231))
    String += get_SpacegroupName(spacegroupNumber)
    isbool = True
    Hist = {}
    idx = 0
    while isbool:
        ele = get_elements()
        wyckoff, wyckoff_data = get_wyckoff(spacegroupNumber)
        xyz = get_xyz(wyckoff_data, wyckoff)
        wyckoff_positions_sub = get_positions(wyckoff_data, wyckoff, xyz)
        coordinates = [coord for array in list(Hist.values()) for coord in array.tolist()]
        result = [np.array_equal(item, hist_item) for hist_item in coordinates for item in wyckoff_positions_sub]
        if not any(result):
            idx += 1
            String += ele + " "
            String += wyckoff + " "
            String += xyz
            Hist[f"{idx}, {ele}"] = wyckoff_positions_sub
            
            isbool = random.choice([True, False])
            
    return String[:-1], spacegroupNumber, Hist

In [8]:
representation, spacegroupNumber, Hist = main()
representation

'P 6 1 1 Co 6d 0.082 0.883 0.055'

### Mutation

In [9]:
def mutation(representation, spacegroupNumber, Hist):
    hist = Hist.copy()
    tokens = representation.split()
    # index = random.randint(0, len(np.array(tokens[4:]).reshape(-1, 5)))
    index = 0
    if index == 0:
        #Get new SpacegroupName
        spacegroupNumber = random.choice(range(1, 231))
        SpacegroupName = get_SpacegroupName(spacegroupNumber)
        tokens[:4] = SpacegroupName.split()
        print(f"Spacegroup Name changed: New Number = {spacegroupNumber}, New Name = {SpacegroupName}")
        
        keylist = list(hist.keys())
        WYInfolist = np.array(tokens[4:]).reshape(-1, 5)
        for idx, i  in enumerate(WYInfolist):
            wyckoff, wyckoff_data = get_wyckoff(spacegroupNumber)
            xyz = get_xyz(wyckoff_data, wyckoff)
            wyckoff_positions_sub = get_positions(wyckoff_data, wyckoff, xyz)
            WYInfolist[idx][1] = wyckoff
            WYInfolist[idx][2:] = xyz.split(" ")[:-1]
            hist[keylist[idx]] = wyckoff_positions_sub
        tokens[4:] = WYInfolist.flatten()
    
    else:
        wy_idx = random.randint(0, 4)
        if wy_idx == 0: 
            #Get new element
            new_element = get_elements()
            while new_element == tokens[4 + (index - 1) * 5 + wy_idx]:
                new_element = get_elements()
            tokens[4 + (index - 1) * 5 + wy_idx] = new_element
    
            #Update history
            old_key = list(hist.keys())[index-1]
            new_key = old_key.split(", ")
            new_key[1] = new_element
            new_key = f"{new_key[0]}, {new_key[1]}"
            hist[new_key] = hist.pop(old_key)
            print(f"Element changed to: {new_element}")
            
        elif wy_idx == 1:
            #Get new wyckoff
            Continue = True
            while Continue:
                new_wyckoff, new_wyckoff_data = get_wyckoff(spacegroupNumber)
                while new_wyckoff == tokens[4 + (index - 1) * 5 + wy_idx]:
                    new_wyckoff, new_wyckoff_data = get_wyckoff(spacegroupNumber)
                xyz = get_xyz(new_wyckoff_data, new_wyckoff)
    
                #Validate representation and update history
                wyckoff_positions_sub = get_positions(new_wyckoff_data, new_wyckoff, xyz)
                coordinates = [coord for array in list(hist.values()) for coord in array.tolist()]
                result = [np.array_equal(item, hist_item) for hist_item in coordinates for item in wyckoff_positions_sub]
                if not any(result):
                    tokens[4 + (index - 1) * 5 + wy_idx+1:4 + (index - 1) * 5 + wy_idx+4] = xyz.split()
                    tokens[4 + (index - 1) * 5 + wy_idx] = new_wyckoff
                    hist[f"{index}, {tokens[4 + (index - 1) * 5]}"] = wyckoff_positions_sub
                    Continue = False
                    print(f"Wyckoff position changed to: {new_wyckoff} with xyz {xyz}")
        else:
            #Get new xyz
            if float(tokens[4 + (index - 1) * 5 + wy_idx]) == 0:
                print("Cannot change XYZ value as X, Y, or Z is not involved.")
            else:
                Continue = True
                while Continue:
                    new_xyz = "0."+"".join(map(str, random.choices(range(10), k=3))) + " "
                    xyzlist = np.array(tokens[4:]).reshape(-1, 5)[index-1]
                    xyzlist[wy_idx] = new_xyz
                    
                    #Validate representation and update history
                    wyckoff_positions_sub = get_positions(get_wyckoff_dict_from_sgn(spacegroupNumber), xyzlist[1], " ".join(xyzlist[2:]))
                    coordinates = [coord for array in list(hist.values()) for coord in array.tolist()]
                    result = [np.array_equal(item, hist_item) for hist_item in coordinates for item in wyckoff_positions_sub]
                    if not any(result):
                        tokens[4 + (index - 1) * 5 + wy_idx] = new_xyz
                        hist[f"{index}, {xyzlist[0]}"] = wyckoff_positions_sub
                        Continue = False
                        print(f"XYZ coordinates changed to: {new_xyz}")
    return " ".join(tokens), spacegroupNumber, hist

In [10]:
representation

'P 6 1 1 Co 6d 0.082 0.883 0.055'

In [11]:
Hist

{'1, Co': array([[0.082, 0.883, 0.055],
        [0.117, 0.199, 0.055],
        [0.801, 0.918, 0.055],
        [0.918, 0.117, 0.055],
        [0.883, 0.801, 0.055],
        [0.199, 0.082, 0.055]])}

In [12]:
new_representation, new_spacegroupNumber, new_hist = mutation(representation, spacegroupNumber, Hist)

Spacegroup Name changed: New Number = 142, New Name = I 41/a c d 


In [13]:
new_representation

'I 41/a c d Co 16d 0.000 0.000 0.767'

In [14]:
new_hist

{'1, Co': array([[0.   , 0.   , 0.767],
        [0.   , 0.5  , 0.017],
        [0.5  , 0.   , 0.483],
        [0.5  , 0.5  , 0.233],
        [0.   , 0.5  , 0.483],
        [0.   , 0.   , 0.233],
        [0.5  , 0.5  , 0.767],
        [0.5  , 0.   , 0.017],
        [0.5  , 0.5  , 0.267],
        [0.5  , 0.   , 0.517],
        [0.   , 0.5  , 0.983],
        [0.   , 0.   , 0.733],
        [0.5  , 0.   , 0.983],
        [0.5  , 0.5  , 0.733],
        [0.   , 0.   , 0.267],
        [0.   , 0.5  , 0.517]])}

### Generate Data

In [15]:
def convert_to_serializable(obj):
    if isinstance(obj, np.ndarray):  # Convert NumPy arrays to lists
        return obj.tolist()
    elif isinstance(obj, list):  # Process each item if it's a list
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, dict):  # Process dictionary values recursively
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    else:
        return obj

In [16]:
DATA = {}
for i in range(30):
    representation, spacegroupNumber, Hist = main()

    Coords = []
    for coord in Hist.values():
        Coords.extend(coord.tolist())
    
    DATA[i] = {
        "REPRESENTATION": representation,
        "SPACEGROUPNUMBER": spacegroupNumber,
        "HIST":convert_to_serializable(Hist)
    }

In [17]:
with open("RandomGenDATA.json", "w") as f:
    json.dump(DATA, f, indent=4)